# 📊 Exploratory Data Analysis
## Text-Based Content Moderation & Toxicity Classifier

**Dataset:** Kaggle Toxic Comment Classification Challenge  
**Goal:** Understand the data distribution, class imbalance, and text characteristics before training.


In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
ACCENT = '#00d4aa'
RED    = '#f85149'

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
from preprocess import load_data, preprocess_dataframe

df_raw = load_data('../data/train.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

## 2. Basic Statistics

In [ ]:
print('=== DataFrame Info ===')
df_raw.info()
print(f'\nMissing values:\n{df_raw.isnull().sum()}')

## 3. Class Distribution

In [ ]:
counts = df_raw['toxic'].value_counts()
labels = ['Non-Toxic', 'Toxic']
colors = [ACCENT, RED]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
ax1.bar(labels, counts.values, color=colors, edgecolor='none', alpha=0.9)
ax1.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax1.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax1.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold', color='white')

# Pie
ax2.pie(counts.values, labels=labels, colors=colors,
        autopct='%1.1f%%', startangle=90,
        textprops={'color': 'white', 'fontsize': 12})
ax2.set_title('Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../static/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Text Length Distribution

In [ ]:
df_raw['char_len']  = df_raw['comment_text'].str.len()
df_raw['word_len']  = df_raw['comment_text'].str.split().str.len()

print('Character length stats by class:')
print(df_raw.groupby('toxic')['char_len'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for cls, color, name in [(0, ACCENT, 'Non-Toxic'), (1, RED, 'Toxic')]:
    subset = df_raw[df_raw['toxic'] == cls]
    axes[0].hist(subset['char_len'].clip(upper=2000), bins=60,
                 alpha=0.6, color=color, label=name)
    axes[1].hist(subset['word_len'].clip(upper=400), bins=60,
                 alpha=0.6, color=color, label=name)

axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].legend()

axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].legend()

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../static/eda_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Preprocessing Preview

In [ ]:
df_clean = preprocess_dataframe(df_raw.copy())
print(f'After preprocessing: {len(df_clean)} rows')
df_clean[['comment_text', 'cleaned_text', 'toxic']].sample(5, random_state=42)

## 6. Top Tokens — Toxic vs Non-Toxic

In [ ]:
from collections import Counter
import re

STOPWORDS = {
    'the','and','for','that','this','you','are','was','with','have',
    'from','your','not','but','they','its','will','can','one','has',
    'what','all','just','said','more','been','when','him','her','also',
    'about','than','into','how','some','then','there','like','would',
    'their','were','which','who','out','get','don','did','got','only',
}

def top_words(texts, n=20):
    words = re.findall(r'\b[a-z]{3,}\b', ' '.join(texts).lower())
    return Counter(w for w in words if w not in STOPWORDS).most_common(n)

toxic_top    = top_words(df_clean[df_clean['toxic']==1]['cleaned_text'])
nontoxic_top = top_words(df_clean[df_clean['toxic']==0]['cleaned_text'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

labels_t, vals_t = zip(*toxic_top)
ax1.barh(labels_t[::-1], vals_t[::-1], color=RED, alpha=0.85)
ax1.set_title('Top 20 Words — Toxic', fontsize=13, fontweight='bold', color=RED)
ax1.spines[['top', 'right']].set_visible(False)

labels_n, vals_n = zip(*nontoxic_top)
ax2.barh(labels_n[::-1], vals_n[::-1], color=ACCENT, alpha=0.85)
ax2.set_title('Top 20 Words — Non-Toxic', fontsize=13, fontweight='bold', color=ACCENT)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('../static/eda_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Takeaways

| Insight | Value |
|---|---|
| Total comments | ~159,571 |
| Toxic comments | ~15,294 (~9.6%) |
| Class imbalance ratio | ~10:1 |
| Avg. words per comment | ~67 |
| Toxic comments slightly longer | Yes |

**Implications for modelling:**
- The dataset is **significantly imbalanced** → use `class_weight='balanced'` in models.
- **Macro F1** is a better metric than raw accuracy.
- Bigrams are likely useful ("hate you" captures toxicity better than single tokens).
